# Intersect QC Bed Files For HG002 

This work is in support of our HG002 QC validation where we run the "HPRC recipe" version of HG002 through our QC in order to determine it's accuracy and the accuracy of our QC pipeline.

**Intersect bed files from Flagger HiFi/ONT and Nucflag to output regions that are likely erronious.**

The final bed file has genomic regions from multiple BED files which were unified by finding when they 
overlap or are within a specified distance (default 100kb) of each other . Regions are only as "error" if 
they contain regions from at least 2 different files. Singletones are labeled as "qc_flag".

Each name field in the output BED encodes whether the region is an error (≥2 supporting QC sources) 
or a qc_flag (only 1 source), followed by a JSON-like list of the supporting evidence.

Below is an example:

```
error{flagger_hifi:MISJOIN|HG002#1#chr1:300-565994;nucflag:DUP|HG002#1#chr1:350-565994}
```

------------------
**The inputs are:**
* Flagger HiFi
* Flagger ONT (r9 and r10)
* Nucflag (HiFi v0.3.3)

This intersection is performed as was done for HPRC R2 assemblies. Comparing the output to GQC results from the assembly compared to Q100 allows us to estimate the accuracy of our intersected QC.

We are running an intersected r9 and r10 separately to simulate the data in Release 2.

In [12]:
import os
import pandas as pd
import boto3
from urllib.parse import urlparse

from bisect import bisect_left
from collections import defaultdict
from typing import List, Tuple, Dict
import gzip

In [13]:
%cd /private/groups/hprc/qc_testing/hg002_qc_comparison

/private/groups/hprc/qc_testing/hg002_qc_comparison


# Get QC Bed Files

## Function Def

In [14]:
def download_s3_files_from_index(df, location_col, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    def download_s3_file(s3_url, local_path):
        parsed = urlparse(s3_url)
        bucket = parsed.netloc
        key = parsed.path.lstrip('/')
        s3 = boto3.client('s3')
        s3.download_file(bucket, key, local_path)

    local_paths = []

    for _, row in df.iterrows():
        s3_url = row.get(location_col)

        # Skip if the path is missing or invalid
        if not isinstance(s3_url, str) or not s3_url.startswith("s3://"):
            local_paths.append(None)
            continue

        filename = os.path.basename(urlparse(s3_url).path)
        local_file = os.path.join(output_dir, filename)

        if not os.path.exists(local_file):
            print(f"Downloading {filename}...")
            download_s3_file(s3_url, local_file)

        local_paths.append(local_file)

    return local_paths


# Build Table

In [15]:
qc_sheet_id = "1Gg3C7jpU9YU9HhTbM5Q63HlyZxflmGHcj_IBjwn5Ky8"
qc_sheet_url = f"https://docs.google.com/spreadsheets/d/{qc_sheet_id}/export?format=csv&gid=0"

# The sheet has a title row and a blank row before the real header.
qc_sheet_raw = pd.read_csv(qc_sheet_url, header=None)
header_row = qc_sheet_raw.index[qc_sheet_raw.iloc[:, 0].eq("file")][0]
qc_files_df = pd.read_csv(qc_sheet_url, skiprows=header_row)

qc_files_df = qc_files_df.dropna(subset=["file", "haplotype", "location"])
qc_files_df = qc_files_df[qc_files_df["haplotype"].str.startswith("hap_")]
qc_files_wide = qc_files_df.pivot(index="haplotype", columns="file", values="location")

qc_combos = {
    "flagger_hifi_flagger_ont_r9_nucflag_hifi_v0.3.3": "flagger_ont_r9",
    "flagger_hifi_flagger_ont_r10_nucflag_hifi_v0.3.3": "flagger_ont_r10",
}

qc_rows = []
for haplotype, files in qc_files_wide.iterrows():
    hap_num = haplotype.replace("hap_", "")

    for qc_combo, flagger_ont_col in qc_combos.items():
        qc_rows.append(
            {
                "assembly_name": f"HG002_{haplotype}_{qc_combo}",
                "sample_id": "HG002",
                "haplotype": hap_num,
                "sample_hap": haplotype,
                "qc_combo": qc_combo,
                "location_flagger_hifi": files["flagger_hifi"],
                "location_flagger_ont": files[flagger_ont_col],
                "location_nucflag": files["nucflag_hifi_v0.3.3"],
            }
        )

In [16]:
qc_index_df = pd.DataFrame(qc_rows)

os.makedirs("qc_indexes", exist_ok=True)
qc_index_df["local_flagger_hifi"] = download_s3_files_from_index(
    qc_index_df,
    "location_flagger_hifi",
    "qc_beds/flagger_hifi",
)
qc_index_df["local_flagger_ont"] = download_s3_files_from_index(
    qc_index_df,
    "location_flagger_ont",
    "qc_beds/flagger_ont",
)
qc_index_df["local_nucflag"] = download_s3_files_from_index(
    qc_index_df,
    "location_nucflag",
    "qc_beds/nucflag_hifi_v0.3.3",
)

qc_index_df


,assembly_name,sample_id,haplotype,sample_hap,qc_combo,location_flagger_hifi,location_flagger_ont,location_nucflag,local_flagger_hifi,local_flagger_ont,local_nucflag
0,HG002_hap_1_flagger_hifi_flagger_ont_r9_nucfla...,HG002,1,hap_1,flagger_hifi_flagger_ont_r9_nucflag_hifi_v0.3.3,s3://human-pangenomics/submissions/DC27718F-5F...,s3://human-pangenomics/submissions/DC27718F-5F...,s3://human-pangenomics/submissions/DC27718F-5F...,qc_beds/flagger_hifi/HG002.hprc_v2.for_genbank...,qc_beds/flagger_ont/HG002.hprc_v2.for_genbank....,qc_beds/nucflag_hifi_v0.3.3/HG002_hprc_v2_hifi...
1,HG002_hap_1_flagger_hifi_flagger_ont_r10_nucfl...,HG002,1,hap_1,flagger_hifi_flagger_ont_r10_nucflag_hifi_v0.3.3,s3://human-pangenomics/submissions/DC27718F-5F...,s3://human-pangenomics/submissions/DC27718F-5F...,s3://human-pangenomics/submissions/DC27718F-5F...,qc_beds/flagger_hifi/HG002.hprc_v2.for_genbank...,qc_beds/flagger_ont/HG002.hprc_v2.for_genbank....,qc_beds/nucflag_hifi_v0.3.3/HG002_hprc_v2_hifi...
2,HG002_hap_2_flagger_hifi_flagger_ont_r9_nucfla...,HG002,2,hap_2,flagger_hifi_flagger_ont_r9_nucflag_hifi_v0.3.3,s3://human-pangenomics/submissions/DC27718F-5F...,s3://human-pangenomics/submissions/DC27718F-5F...,s3://human-pangenomics/submissions/DC27718F-5F...,qc_beds/flagger_hifi/HG002.hprc_v2.for_genbank...,qc_beds/flagger_ont/HG002.hprc_v2.for_genbank....,qc_beds/nucflag_hifi_v0.3.3/HG002_hprc_v2_hifi...
3,HG002_hap_2_flagger_hifi_flagger_ont_r10_nucfl...,HG002,2,hap_2,flagger_hifi_flagger_ont_r10_nucflag_hifi_v0.3.3,s3://human-pangenomics/submissions/DC27718F-5F...,s3://human-pangenomics/submissions/DC27718F-5F...,s3://human-pangenomics/submissions/DC27718F-5F...,qc_beds/flagger_hifi/HG002.hprc_v2.for_genbank...,qc_beds/flagger_ont/HG002.hprc_v2.for_genbank....,qc_beds/nucflag_hifi_v0.3.3/HG002_hprc_v2_hifi...


# Intersect BEDs

## Function Def

In [17]:
# ----------------- Types -----------------
Interval = Tuple[int, int, str, int]  # start, end, evidence_string, file_idx
ChromMap = Dict[str, List[Tuple[int, int, str]]]  # start, end, evidence_string


# ----------------- I/O helpers -----------------
def open_maybe_gz(path: str):
    return gzip.open(path, "rt") if path.endswith(".gz") else open(path, "r")


def read_bed(path: str, evidence_type: str) -> ChromMap:
    """
    Read a BED (3+ columns). Use col4 as 'call' when present; otherwise 'NA'.
    Build evidence string: evidence_type:call|chrom:start-end
    """
    by_chr: ChromMap = defaultdict(list)
    with open_maybe_gz(path) as fh:
        for line in fh:
            if (not line.strip() or
                line.startswith("#") or
                line.startswith("track") or
                line.startswith("browser")):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            chrom, s, e = parts[0], parts[1], parts[2]
            try:
                start, end = int(s), int(e)
            except ValueError:
                continue
            if end <= start:
                continue
            call = parts[3] if len(parts) >= 4 and parts[3] else "NA"
            region = f"{chrom}:{start}-{end}"
            evidence = f"{evidence_type}:{call}|{region}"
            by_chr[chrom].append((start, end, evidence))
    # sort each chrom by start
    for chrom in by_chr:
        by_chr[chrom].sort(key=lambda x: x[0])
    return by_chr


def within_distance(a: Tuple[int, int], b: Tuple[int, int], dist: int) -> bool:
    (a_s, a_e), (b_s, b_e) = a, b
    if a_e <= b_s:
        gap = b_s - a_e
    elif b_e <= a_s:
        gap = a_s - b_e
    else:
        gap = 0  # overlap
    return gap <= dist


# ----------------- DSU (Union-Find) -----------------
class DSU:
    def __init__(self, n: int):
        self.p = list(range(n))
        self.r = [0]*n

    def find(self, x: int) -> int:
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x

    def union(self, a: int, b: int):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.r[ra] < self.r[rb]:
            self.p[ra] = rb
        elif self.r[ra] > self.r[rb]:
            self.p[rb] = ra
        else:
            self.p[rb] = ra
            self.r[ra] += 1


# ----------------- Core processing -----------------
def write_consensus_err_bed(
    bed_files: List[str],
    evidence_types: List[str],
    distance: int = 100_000,
    output: str = "unified_qc.bed"
):
    """
    bed_files: list of BED paths (3+)
    evidence_types: same length as bed_files; e.g., ["flagger_hifi","flagger_ont","nucflag"]
    distance: merge window in bp
    output: BED9 with name=JSON-like evidence list, itemRgb red(error) / orange(qc_flag)
    """

    assert len(bed_files) >= 3, "Please provide at least 3 BED files."
    assert len(bed_files) == len(evidence_types), "evidence_types must match bed_files length."

    # Read all beds
    beds = [read_bed(p, et) for p, et in zip(bed_files, evidence_types)]

    # Collect per-chrom all intervals with file_idx
    by_chr_all: Dict[str, List[Interval]] = defaultdict(list)
    for file_idx, bed in enumerate(beds):
        for chrom, ivs in bed.items():
            for s, e, ev in ivs:
                by_chr_all[chrom].append((s, e, ev, file_idx))

    # For each chrom, union intervals that are within_distance
    merged_outputs = []  # list of (chrom, start, end, status, evidences)

    for chrom, records in by_chr_all.items():
        # sort by start for sweep
        records.sort(key=lambda x: x[0])
        n = len(records)
        dsu = DSU(n)
        starts = [r[0] for r in records]

        # Sweep-line: for each i, compare to j forward until start_j > end_i + dist
        for i in range(n):
            i_s, i_e = records[i][0], records[i][1]
            # binary search start bound
            j = i + 1
            # advance while starts[j] within window
            while j < n and starts[j] <= i_e + distance:
                if within_distance((i_s, i_e), (records[j][0], records[j][1]), distance):
                    dsu.union(i, j)
                j += 1

        # Gather components
        comp_members: Dict[int, List[int]] = defaultdict(list)
        for i in range(n):
            comp_members[dsu.find(i)].append(i)

        for root, idxs in comp_members.items():
            comp_start = min(records[i][0] for i in idxs)
            comp_end   = max(records[i][1] for i in idxs)
            evidences  = [records[i][2] for i in idxs]
            file_ids   = {records[i][3] for i in idxs}
            status = "error" if len(file_ids) >= 2 else "qc_flag"

            # de-duplicate while preserving order
            seen = set()
            ev_list = []
            for ev in evidences:
                if ev not in seen:
                    seen.add(ev)
                    ev_list.append(ev)

            merged_outputs.append((chrom, comp_start, comp_end, status, ev_list))

    # Write unified BED9
    merged_outputs.sort(key=lambda x: (x[0], x[1], x[2]))
    with open(output, "w") as out:
        for chrom, start, end, status, ev_list in merged_outputs:
            name = f"{status}{{" + ";".join(ev_list) + "}}"
            score = 0
            strand = "."
            thickStart = start
            thickEnd = end
            itemRgb = "255,0,0" if status == "error" else "255,255,0"  # red / yellow
            out.write(
                f"{chrom}\t{start}\t{end}\t{name}\t{score}\t{strand}\t{thickStart}\t{thickEnd}\t{itemRgb}\n"
            )



## Run Intersection

In [18]:
os.makedirs("consensus_errors", exist_ok=True)

## Keep file paths that we are creating (to add back to df)
consensus_paths = []

## For each assembly, pass QC beds and find regions that
## have overlapping or nearby QC flags. Requires that there is
## an overlap or nearby hit from ANOTHER file.
for i, row in qc_index_df.iterrows():
    bed_files = [
        row["local_nucflag"],
        row["local_flagger_hifi"],
        row["local_flagger_ont"],
    ]
    evidence_types = ["nucflag", "flagger_hifi", "flagger_ont"]

    out_file = os.path.join(
        "consensus_errors",
        f"{row['assembly_name']}_consensus_errors_v0.1.bed"
    )

    write_consensus_err_bed(bed_files, evidence_types, distance=100000, output=out_file)

    consensus_paths.append(out_file)

qc_index_df["local_consensus_err_bed"] = consensus_paths

In [19]:
qc_index_df.to_csv("qc_indexes/combined_qc_index_locs.csv", index=False)

# Upload Intersected QC Beds

In [24]:
! mkdir -p /private/groups/hprc/qc_testing/hg002_qc_comparison/qc_beds/intersected_qc/upload/HG002/assemblies/freeze_2/assembly_pipeline/ncbi_upload/assembly_qc/intersected_qc/

In [25]:
! cp \
    /private/groups/hprc/qc_testing/hg002_qc_comparison/consensus_errors/* \
    /private/groups/hprc/qc_testing/hg002_qc_comparison/qc_beds/intersected_qc/upload/HG002/assemblies/freeze_2/assembly_pipeline/ncbi_upload/assembly_qc/intersected_qc/

In [26]:
! cp \
    /private/groups/hprc/qc_testing/hg002_qc_comparison/consensus_errors/* \
    /private/groups/hprc/qc_testing/hg002_qc_comparison/qc_beds/intersected_qc/

In [27]:
%cd /private/groups/hprc/qc_testing/hg002_qc_comparison/qc_beds/intersected_qc/

/private/groups/hprc/qc_testing/hg002_qc_comparison/qc_beds/intersected_qc


In [31]:
# %%bash 
# ssds staging upload \
#     --submission-id DC27718F-5F38-43B0-9A78-270F395F13E8 \
#     upload \
#     &>>hg002_intersected_qc_s3_upload.stderr